# IDK-1 — Step 2: Data Pipeline

Prepare training data yang bersih → binary `.bin` format.

**Target:** 2-3B token bersih dari Wikipedia ID + CulturaX ID

**Setup Kaggle — tambahkan dataset sebagai input:**
- Tokenizer: `ripkii/idk1-tokenizer`

**Output:**
- `train.bin` + `val.bin` → upload ke Kaggle dataset `idk1-data`

In [ ]:
!pip install -q datasets tokenizers

In [ ]:
import os
import re
import json
import hashlib
import numpy as np
from datasets import load_dataset
from tokenizers import Tokenizer
from collections import Counter

print("Libraries OK")

## 1. Config

In [ ]:
# ── Path tokenizer ──────────────────────────────────────────────────
# Kaggle:
TOKENIZER_PATH = "/kaggle/input/datasets/ripkii/idk-1-tokenizer/tokenizer.json"
# SageMaker: ganti ke "/home/studio-lab-user/idk1-tokenizer/tokenizer.json"

# ── Output ──────────────────────────────────────────────────────────
OUT_DIR   = "/kaggle/working"
TRAIN_BIN = os.path.join(OUT_DIR, "train.bin")
VAL_BIN   = os.path.join(OUT_DIR, "val.bin")

# ── Cleaning thresholds ─────────────────────────────────────────────
MIN_CHARS        = 100    # dokumen < 100 char dibuang
MIN_WORDS        = 20     # dokumen < 20 kata dibuang
MAX_REPEAT_RATIO = 0.3    # kalau >30% kata adalah duplikat → buang
MAX_LINE_RATIO   = 0.5    # kalau >50% baris adalah short lines → buang

# ── Split ───────────────────────────────────────────────────────────
VAL_RATIO = 0.01          # 1% untuk validasi

print("Config OK")
print(f"Output: {TRAIN_BIN}")

## 2. Load Tokenizer

In [ ]:
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer.enable_truncation(max_length=100_000)  # safety cap

BOS_ID = tokenizer.token_to_id("<s>")
EOS_ID = tokenizer.token_to_id("</s>")

print(f"Tokenizer loaded: vocab={tokenizer.get_vocab_size():,}")
print(f"BOS={BOS_ID}, EOS={EOS_ID}")

## 3. Cleaning Functions

Ini core dari pipeline — filter yang buat IDK-1 lebih bersih dari DFD-1.

In [ ]:
# Pola navigation/boilerplate yang sering muncul di web crawl
NAV_PATTERNS = [
    r".{0,30}\s*[>|»]\s*.{0,30}[>|»]",   # Beranda > Kategori > Artikel
    r"(Home|Beranda|Menu)\s*[|\-]\s*",    # Home | About | Contact
    r"Halaman\s+\d+\s*(dari|of)\s*\d+",   # Halaman 1 dari 10
    r"Tags?\s*:\s*(\w+\s*,\s*){2,}",      # Tags: berita, indonesia, terkini
    r"(Baca|Lihat)\s+(juga|selengkapnya)", # Baca juga / Lihat selengkapnya
    r"Klik\s+(di sini|disini)",            # Klik di sini
    r"Copyright\s+©",                      # Copyright footer
    r"All rights reserved",
    r"Privacy Policy|Terms of Service",
    r"\d{1,2}/\d{1,2}/\d{4}\s+\d{1,2}:\d{2}",  # timestamp spam
]
NAV_REGEX = [re.compile(p, re.IGNORECASE) for p in NAV_PATTERNS]


def is_navigation_noise(text: str) -> bool:
    """True kalau dokumen dominan berisi boilerplate navigasi web."""
    lines = text.split("\n")
    noise_lines = 0
    for line in lines:
        if any(p.search(line) for p in NAV_REGEX):
            noise_lines += 1
    return len(lines) > 0 and (noise_lines / len(lines)) > 0.4


def has_high_repetition(text: str) -> bool:
    """True kalau >30% kata adalah kata yang sama berulang."""
    words = text.lower().split()
    if len(words) < 20:
        return False
    counts = Counter(words)
    most_common_count = counts.most_common(1)[0][1]
    return (most_common_count / len(words)) > MAX_REPEAT_RATIO


def has_too_many_short_lines(text: str) -> bool:
    """True kalau >50% baris sangat pendek (< 30 char) — ciri khas menu/list noise."""
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if len(lines) < 5:
        return False
    short = sum(1 for l in lines if len(l) < 30)
    return (short / len(lines)) > MAX_LINE_RATIO


def is_too_short(text: str) -> bool:
    return len(text) < MIN_CHARS or len(text.split()) < MIN_WORDS


def clean_text(text: str) -> str:
    """Normalisasi whitespace dan karakter aneh."""
    text = re.sub(r"\r\n", "\n", text)
    text = re.sub(r"\t", " ", text)
    text = re.sub(r" {3,}", "  ", text)      # max 2 spasi berturut-turut
    text = re.sub(r"\n{4,}", "\n\n\n", text)  # max 3 newline berturut-turut
    text = text.strip()
    return text


def should_keep(text: str) -> bool:
    """True kalau dokumen lolos semua filter."""
    if is_too_short(text):            return False
    if is_navigation_noise(text):     return False
    if has_high_repetition(text):     return False
    if has_too_many_short_lines(text): return False
    return True


print("Cleaning functions OK")

# Quick test
test_noise = "Beranda > Kategori > Sub > Artikel\nBeranda > Lain > Sub\nClick here\nTags: a, b, c, d, e"
test_good  = "Indonesia adalah negara kepulauan terbesar di dunia yang terletak di Asia Tenggara. Negara ini terdiri dari lebih dari 17.000 pulau."
print(f"Noise doc keep: {should_keep(test_noise)} (harusnya False)")
print(f"Good doc keep:  {should_keep(test_good)} (harusnya True)")

## 4. Deduplication

In [ ]:
def get_doc_hash(text: str) -> str:
    """Hash MD5 dari 200 karakter pertama — cukup untuk exact dedup."""
    return hashlib.md5(text[:200].encode()).hexdigest()


class Deduplicator:
    def __init__(self):
        self.seen = set()

    def is_duplicate(self, text: str) -> bool:
        h = get_doc_hash(text)
        if h in self.seen:
            return True
        self.seen.add(h)
        return False

    def reset(self):
        self.seen.clear()


print("Deduplicator OK")

## 5. Tokenize & Write Binary

Format output: flat array `uint16` — satu file = semua token ID berurutan.
Setiap dokumen diawali `<s>` dan diakhiri `</s>`.

In [ ]:
def process_and_write(dataset_iter, out_path: str, dedup: Deduplicator, 
                      text_field: str = "text", max_docs: int = None):
    """
    Stream dokumen → clean → tokenize → tulis ke binary file.
    Return: (total_docs_processed, total_docs_kept, total_tokens)
    """
    total, kept, tokens = 0, 0, 0
    buffer = []
    FLUSH_EVERY = 50_000  # flush ke disk setiap 50k dokumen

    with open(out_path, "wb") as f:
        for example in dataset_iter:
            total += 1
            text = example.get(text_field, "") or ""

            # Clean
            text = clean_text(text)

            # Filter
            if not should_keep(text):
                continue
            if dedup.is_duplicate(text):
                continue

            # Tokenize
            enc = tokenizer.encode(text)
            ids = [BOS_ID] + enc.ids + [EOS_ID]

            buffer.extend(ids)
            kept  += 1
            tokens += len(ids)

            # Flush buffer ke disk
            if len(buffer) >= FLUSH_EVERY * 512:  # ~25M tokens per flush
                arr = np.array(buffer, dtype=np.uint16)
                f.write(arr.tobytes())
                buffer = []

            if total % 100_000 == 0:
                print(f"  Processed {total:,} | Kept {kept:,} ({kept/total*100:.1f}%) | Tokens {tokens/1e9:.2f}B")

            if max_docs and total >= max_docs:
                break

        # Flush sisa
        if buffer:
            arr = np.array(buffer, dtype=np.uint16)
            f.write(arr.tobytes())

    return total, kept, tokens


print("Process function OK")

## 6. Load & Process Wikipedia ID

In [ ]:
print("Loading Wikipedia ID...")
wiki = load_dataset(
    "wikimedia/wikipedia",
    "20231101.id",
    split="train",
    streaming=True,
    trust_remote_code=True,
)

dedup = Deduplicator()
wiki_tmp = os.path.join(OUT_DIR, "wiki_tmp.bin")

print("Processing Wikipedia ID...")
total, kept, tokens = process_and_write(
    wiki, wiki_tmp, dedup, text_field="text"
)

print(f"\n── Wikipedia ID selesai ──")
print(f"Total dokumen : {total:,}")
print(f"Dokumen kept  : {kept:,} ({kept/total*100:.1f}%)")
print(f"Total tokens  : {tokens/1e9:.2f}B")
print(f"File size     : {os.path.getsize(wiki_tmp)/1e9:.2f} GB")

## 7. Load & Process CulturaX ID

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("HF Login OK!")

In [ ]:
print("Loading CulturaX ID...")
culturax = load_dataset(
    "uonlp/CulturaX",
    "id",
    split="train",
    streaming=True,
    trust_remote_code=True,
)

culturax_tmp = os.path.join(OUT_DIR, "culturax_tmp.bin")

print("Processing CulturaX ID...")
# CulturaX sangat besar — limit dulu untuk balance dengan Wikipedia
# Wikipedia ~700k artikel, CulturaX bisa puluhan juta dokumen
MAX_CULTURAX = 5_000_000  # 5 juta dokumen, ambil yang lolos filter

total_cx, kept_cx, tokens_cx = process_and_write(
    culturax, culturax_tmp, dedup,  # dedup lanjut — hindari overlap dengan Wikipedia
    text_field="text",
    max_docs=MAX_CULTURAX,
)

print(f"\n── CulturaX ID selesai ──")
print(f"Total dokumen : {total_cx:,}")
print(f"Dokumen kept  : {kept_cx:,} ({kept_cx/total_cx*100:.1f}%)")
print(f"Total tokens  : {tokens_cx/1e9:.2f}B")
print(f"File size     : {os.path.getsize(culturax_tmp)/1e9:.2f} GB")

## 8. Merge → train.bin + val.bin

In [ ]:
def merge_and_split(file_paths: list, train_path: str, val_path: str, val_ratio: float = 0.01):
    """
    Merge semua .bin file → interleave chunks antar-source → split train/val.
    Interleave dilakukan di level CHUNK (bukan token) — urutan token dalam dokumen tetap utuh.
    Wikipedia dan CulturaX di-mix secara round-robin per 1M token chunk.
    """
    CHUNK_SIZE = 1_000_000  # 1M token per chunk = 2MB per read

    # Hitung total tokens
    total_tokens = sum(os.path.getsize(fp) // 2 for fp in file_paths)
    val_tokens   = int(total_tokens * val_ratio)
    train_tokens = total_tokens - val_tokens
    print(f"Total tokens : {total_tokens/1e9:.2f}B")
    print(f"Train tokens : {train_tokens/1e9:.2f}B")
    print(f"Val tokens   : {val_tokens/1e9:.2f}B")

    written = 0
    file_handles = [open(fp, "rb") for fp in file_paths]

    with open(train_path, "wb") as f_train, open(val_path, "wb") as f_val:
        active = list(range(len(file_paths)))
        while active:
            next_active = []
            for src_idx in active:
                chunk_bytes = file_handles[src_idx].read(CHUNK_SIZE * 2)  # uint16 = 2 bytes
                if not chunk_bytes:
                    continue
                next_active.append(src_idx)

                chunk = np.frombuffer(chunk_bytes, dtype=np.uint16)
                chunk_len = len(chunk)

                if written + chunk_len <= train_tokens:
                    f_train.write(chunk.tobytes())
                elif written >= train_tokens:
                    f_val.write(chunk.tobytes())
                else:
                    split = train_tokens - written
                    f_train.write(chunk[:split].tobytes())
                    f_val.write(chunk[split:].tobytes())

                written += chunk_len
            active = next_active

    for fh in file_handles:
        fh.close()

    print(f"\ntrain.bin : {os.path.getsize(train_path)/1e9:.2f} GB")
    print(f"val.bin   : {os.path.getsize(val_path)/1e9:.2f} GB")


merge_and_split(
    [wiki_tmp, culturax_tmp],
    TRAIN_BIN, VAL_BIN,
    val_ratio=VAL_RATIO,
)

# Hapus tmp files
os.remove(wiki_tmp)
os.remove(culturax_tmp)
print("\nDone! Tmp files removed.")

## 9. Verifikasi Output

In [ ]:
# Cek train.bin
train_data = np.fromfile(TRAIN_BIN, dtype=np.uint16)
val_data   = np.fromfile(VAL_BIN,   dtype=np.uint16)

print(f"train.bin : {len(train_data)/1e9:.2f}B tokens")
print(f"val.bin   : {len(val_data)/1e6:.1f}M tokens")

print(f"\nMax token ID: {train_data.max()} (harus < 40000)")
print(f"Min token ID: {train_data.min()}")

# Sanity check
assert train_data.max() < 40_000, "Ada token ID di luar vocab!"
assert len(train_data) > 1_000_000, "Training data terlalu sedikit!"
print("\nSanity check PASSED")

# Coherence check — decode beberapa window 512 token
# Kalau data bener, harusnya terbaca sebagai teks Indo yang koheren
print("\n── Coherence Check (3 sample windows) ──")
for i, start in enumerate([0, len(train_data)//4, len(train_data)//2]):
    window = train_data[start:start+512].tolist()
    decoded = tokenizer.decode(window)
    print(f"\n[Window {i+1} @ token {start:,}]")
    print(decoded[:300])
    print("...")

## 10. Upload ke Kaggle Dataset

Setelah cell ini selesai, `train.bin` dan `val.bin` tersimpan di `/kaggle/working/`.
Upload sebagai Kaggle dataset baru bernama `idk1-data`.

In [ ]:
# Summary final
train_gb = os.path.getsize(TRAIN_BIN) / 1e9
val_gb   = os.path.getsize(VAL_BIN)   / 1e9
total_gb = train_gb + val_gb

print("══════════════════════════════")
print("       DATA PIPELINE DONE     ")
print("══════════════════════════════")
print(f"train.bin : {train_gb:.2f} GB  ({len(train_data)/1e9:.2f}B tokens)")
print(f"val.bin   : {val_gb:.3f} GB  ({len(val_data)/1e6:.1f}M tokens)")
print(f"Total     : {total_gb:.2f} GB")
print()
print("Next step:")
print("1. Download train.bin + val.bin dari /kaggle/working/")
print("2. Upload ke Kaggle dataset baru → nama: idk1-data")
print("3. Lanjut ke notebook 03_architecture.ipynb")